# The Main Algorithm , which leverages Semantic & Similarity-Based Architecture for Mentee - Mentor Matchingmaking

In [16]:
import re

def extract_keywords(data):
    """
    Recursively extracts and normalizes all words from a dictionary or list.
    This makes the algorithm immune to schema changes.
    """
    words = []
    if isinstance(data, dict):
      for key, value in data.items():
          # Skip keys that aren't useful for semantic matching
          if key not in ["category", "hindsight_sentiment", "name", "mentor_id", "student_id"]:
              words.extend(extract_keywords(value))
    elif isinstance(data, list):
      for item in data:
          words.extend(extract_keywords(item))
    elif isinstance(data, str):
      # Tokenize, lowercase, and remove basic punctuation
      clean_words = re.findall(r'\b[a-zA-Z]{3,}\b', data.lower())
      words.extend(clean_words)
    return set(words)

def get_risk_score(risk_string):
    """Normalizes risk tolerance to a numerical scale for distance calculation."""
    risk = str(risk_string).lower()
    if "high" in risk: return 3
    if "medium" in risk: return 2
    if "low" in risk: return 1
    return 2 # Default to medium if unspecified

def calculate_dynamic_match_score(mentee, mentor):
    score = 0
    weights = {
        "category_match": 40,
        "keyword_similarity": 35,
        "psychological_sync": 15,
        "structural_empathy": 10
    }

    # ---------------------------------------------------------
    # VECTOR 1: Historical Category Match (Dynamic Lookup)
    # ---------------------------------------------------------
    mentee_dilemma_cat = mentee.get("current_dilemma", {}).get("category", "")
    mentor_history = mentor.get("historical_dilemmas", [])

    # Check if the mentor ever faced this exact category, regardless of what it's named
    matching_dilemma = next((d for d in mentor_history if d.get("category") == mentee_dilemma_cat), None)

    if matching_dilemma:
        score += weights["category_match"]

        # Bonus: Did they regret it or learn from it? We boost for strong sentiments.
        sentiment = matching_dilemma.get("hindsight_sentiment", "").lower()
        if "positive" in sentiment or "negative" in sentiment:
            score += 5 # Boost for having a definitive, actionable outcome

    # ---------------------------------------------------------
    # VECTOR 2: Keyword Jaccard Similarity (Schema-Agnostic)
    # ---------------------------------------------------------
    # Extract ALL context from mentee's current dilemma
    mentee_context = mentee.get("current_dilemma", {}).get("context_specifics", {})
    mentee_tags = extract_keywords(mentee_context)

    # Extract ALL professional and legacy keywords from mentor
    mentor_profile = mentor.get("current_professional_profile", {})
    mentor_legacy = mentor.get("campus_legacy", {})

    mentor_tags = extract_keywords(mentor_profile).union(extract_keywords(mentor_legacy))

    # Calculate Jaccard-like overlap: (Intersection) / (Size of Mentee's needs)
    if len(mentee_tags) > 0:
        intersection = mentee_tags.intersection(mentor_tags)
        overlap_ratio = len(intersection) / len(mentee_tags)
        # Cap the ratio at 1.0, multiply by the weight
        score += min(overlap_ratio * 1.5, 1.0) * weights["keyword_similarity"]

    # ---------------------------------------------------------
    # VECTOR 3: Psychological Proximity
    # ---------------------------------------------------------
    mentee_risk = get_risk_score(mentee.get("current_dilemma", {}).get("psychology", {}).get("risk_tolerance", ""))
    mentor_style = mentor.get("mentoring_meta", {}).get("advice_style", "")
    mentor_risk = get_risk_score(mentor_style)

    # Calculate absolute distance. Distance of 0 = perfect match. Distance of 2 = polar opposites.
    distance = abs(mentee_risk - mentor_risk)
    if distance == 0:
        score += weights["psychological_sync"]
    elif distance == 1:
        score += (weights["psychological_sync"] * 0.5) # Partial points for being close
    # Distance of 2 gets 0 points for psychology.

    # ---------------------------------------------------------
    # VECTOR 4: Structural Empathy (Branch & CGPA)
    # ---------------------------------------------------------
    mentee_profile = mentee.get("profile", {})
    mentor_legacy = mentor.get("campus_legacy", {})

    if mentee_profile.get("branch") == mentor_legacy.get("branch"):
        score += (weights["structural_empathy"] * 0.5)

    if mentee_profile.get("cgpa_bracket") == mentor_legacy.get("cgpa_at_graduation"):
        score += (weights["structural_empathy"] * 0.5)

    return round(min(score, 100), 2)

# --- Execution Function ---
def get_top_mentors(mentee_json, all_mentors_json, top_n=5):
    scored_mentors = []

    for mentor in all_mentors_json:
        score = calculate_dynamic_match_score(mentee_json, mentor)
        scored_mentors.append({
            "mentor_id": mentor["mentor_id"],
            "mentor_name": mentor["current_professional_profile"]["name"],
            "current_role": mentor["current_professional_profile"]["current_role"],
            "match_score": score
        })

    # Sort descending by score
    scored_mentors.sort(key=lambda x: x["match_score"], reverse=True)
    return scored_mentors[:top_n]

With the architecture designed , we run this alogrithm on 4 test mentees, namely each from different year.

# YEAR 1 Mentee

In [17]:
import json

# Loading the data
with open('mentee_y1.json', 'r') as file:
    mentee_y1_data = json.load(file)

with open('mock_mentors.json', 'r') as file:
    all_mentors_data = json.load(file)

# Running the main algorithm
top_matches = get_top_mentors(mentee_y1_data, all_mentors_data)

mentee_name = mentee_y1_data["profile"]["name"]
academic_year = mentee_y1_data["profile"]["academic_year"]

print(f"\n TOP MENTOR MATCHES FOR: {mentee_name.upper()} (YEAR {academic_year})")
print("=" * 65)

for rank, mentor in enumerate(top_matches, start=1):
    name = mentor.get('mentor_name', 'Unknown')
    role = mentor.get('current_role', 'Unknown Role')
    score = mentor.get('match_score', 0)

    print(f" Rank {rank}: {name}")
    print(f"    Role  : {role}")
    print(f"    Score : {score} / 100")
    print("-" * 65)


 TOP MENTOR MATCHES FOR: KABIR (YEAR 1)
 Rank 1: Sanya
    Role  : Data Scientist
    Score : 61.31 / 100
-----------------------------------------------------------------
 Rank 2: Ishaan
    Role  : Quantitative Analyst
    Score : 57.75 / 100
-----------------------------------------------------------------
 Rank 3: Neha
    Role  : Software Development Engineer
    Score : 25.25 / 100
-----------------------------------------------------------------
 Rank 4: Kavya
    Role  : Senior Business Analyst
    Score : 24.19 / 100
-----------------------------------------------------------------
 Rank 5: Aditi
    Role  : Product Manager II
    Score : 24.19 / 100
-----------------------------------------------------------------


# YEAR 2 Mentee


In [18]:
# Loading the data
with open('mentee_y2.json', 'r') as file:
    mentee_y2_data = json.load(file)

with open('mock_mentors.json', 'r') as file:
    all_mentors_data = json.load(file)

# Running the main algorithm
top_matches = get_top_mentors(mentee_y2_data, all_mentors_data)

mentee_name = mentee_y2_data["profile"]["name"]
academic_year = mentee_y2_data["profile"]["academic_year"]

print(f"\n TOP MENTOR MATCHES FOR: {mentee_name.upper()} (YEAR {academic_year})")
print("=" * 65)

for rank, mentor in enumerate(top_matches, start=1):
    name = mentor.get('mentor_name', 'Unknown')
    role = mentor.get('current_role', 'Unknown Role')
    score = mentor.get('match_score', 0)

    print(f" Rank {rank}: {name}")
    print(f"    Role  : {role}")
    print(f"    Score : {score} / 100")
    print("-" * 65)


 TOP MENTOR MATCHES FOR: RIYA (YEAR 2)
 Rank 1: Sanya
    Role  : Data Scientist
    Score : 58.5 / 100
-----------------------------------------------------------------
 Rank 2: Neha
    Role  : Software Development Engineer
    Score : 47.5 / 100
-----------------------------------------------------------------
 Rank 3: Siddharth
    Role  : Product Designer
    Score : 17.5 / 100
-----------------------------------------------------------------
 Rank 4: Aditi
    Role  : Product Manager II
    Score : 12.5 / 100
-----------------------------------------------------------------
 Rank 5: Kavya
    Role  : Senior Business Analyst
    Score : 7.5 / 100
-----------------------------------------------------------------


# Year 3 Mentee

In [19]:
# Loading the data
with open('mentee_y3.json', 'r') as file:
    mentee_y3_data = json.load(file)

with open('mock_mentors.json', 'r') as file:
    all_mentors_data = json.load(file)

# Running the main algorithm
top_matches = get_top_mentors(mentee_y3_data, all_mentors_data)

mentee_name = mentee_y3_data["profile"]["name"]
academic_year = mentee_y3_data["profile"]["academic_year"]

print(f"\n TOP MENTOR MATCHES FOR: {mentee_name.upper()} (YEAR {academic_year})")
print("=" * 65)

for rank, mentor in enumerate(top_matches, start=1):
    name = mentor.get('mentor_name', 'Unknown')
    role = mentor.get('current_role', 'Unknown Role')
    score = mentor.get('match_score', 0)

    print(f" Rank {rank}: {name}")
    print(f"    Role  : {role}")
    print(f"    Score : {score} / 100")
    print("-" * 65)


 TOP MENTOR MATCHES FOR: ANANYA (YEAR 3)
 Rank 1: Aryan
    Role  : Founder & Technical Lead
    Score : 60.5 / 100
-----------------------------------------------------------------
 Rank 2: Vikram
    Role  : Silicon Design Engineer
    Score : 59.0 / 100
-----------------------------------------------------------------
 Rank 3: Meera
    Role  : Policy Fellow / Founder
    Score : 24.5 / 100
-----------------------------------------------------------------
 Rank 4: Sanya
    Role  : Data Scientist
    Score : 21.5 / 100
-----------------------------------------------------------------
 Rank 5: Neha
    Role  : Software Development Engineer
    Score : 18.0 / 100
-----------------------------------------------------------------


# Year 4 Mentee

In [20]:
# Loading the data
with open('mentee_y4.json', 'r') as file:
    mentee_y4_data = json.load(file)

with open('mock_mentors.json', 'r') as file:
    all_mentors_data = json.load(file)

# Running the main algorithm
top_matches = get_top_mentors(mentee_y4_data, all_mentors_data)

mentee_name = mentee_y4_data["profile"]["name"]
academic_year = mentee_y4_data["profile"]["academic_year"]

print(f"\n TOP MENTOR MATCHES FOR: {mentee_name.upper()} (YEAR {academic_year})")
print("=" * 65)

for rank, mentor in enumerate(top_matches, start=1):
    name = mentor.get('mentor_name', 'Unknown')
    role = mentor.get('current_role', 'Unknown Role')
    score = mentor.get('match_score', 0)

    print(f" Rank {rank}: {name}")
    print(f"    Role  : {role}")
    print(f"    Score : {score} / 100")
    print("-" * 65)


 TOP MENTOR MATCHES FOR: DHRUV (YEAR 4)
 Rank 1: Aditi
    Role  : Product Manager II
    Score : 59.31 / 100
-----------------------------------------------------------------
 Rank 2: Kavya
    Role  : Senior Business Analyst
    Score : 52.5 / 100
-----------------------------------------------------------------
 Rank 3: Aryan
    Role  : Founder & Technical Lead
    Score : 20.0 / 100
-----------------------------------------------------------------
 Rank 4: Ishaan
    Role  : Quantitative Analyst
    Score : 18.62 / 100
-----------------------------------------------------------------
 Rank 5: Vikram
    Role  : Silicon Design Engineer
    Score : 18.62 / 100
-----------------------------------------------------------------
